# Malware Analysis and Reverse Engineering

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import hashlib
import zlib
import base64
import struct
from collections import Counter
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
import re

class MalwareAnalyzer:
    def __init__(self):
        self.pe_signatures = {}
        self.behavior_patterns = []
        self.api_calls = []
        self.control_flow_graph = nx.DiGraph()

    def generate_malware_sample(self, malware_type="trojan"):
        templates = {
            "trojan": b"\x4d\x5a\x90\x00" + b"CreateProcess" * 10 + b"RegSetValue" * 8,
            "ransomware": b"\x4d\x5a\x90\x00" + b"CryptEncrypt" * 15 + b"FindFirstFile" * 12,
            "rootkit": b"\x4d\x5a\x90\x00" + b"SetWindowsHook" * 10 + b"NtQuerySystem" * 8,
            "botnet": b"\x4d\x5a\x90\x00" + b"WSAStartup" * 12 + b"InternetConnect" * 10
        }

        base_sample = templates.get(malware_type, templates["trojan"])
        obfuscated = self.apply_obfuscation(base_sample)
        return obfuscated, malware_type

    def apply_obfuscation(self, data):
        xor_key = 0xAA
        obfuscated = bytes([b ^ xor_key for b in data])

        packed = zlib.compress(obfuscated)
        encoded = base64.b64encode(packed)

        return encoded

    def static_analysis(self, sample_data):
        entropy = self.calculate_entropy(sample_data)

        try:
            decoded = base64.b64decode(sample_data)
            decompressed = zlib.decompress(decoded)
            deobfuscated = bytes([b ^ 0xAA for b in decompressed])
        except:
            deobfuscated = sample_data

        strings = self.extract_strings(deobfuscated)
        imports = self.extract_imports(deobfuscated)
        pe_header = self.parse_pe_header(deobfuscated)

        return {
            "entropy": entropy,
            "strings": strings,
            "imports": imports,
            "pe_header": pe_header,
            "file_size": len(sample_data)
        }

    def calculate_entropy(self, data):
        if not data:
            return 0
        counts = Counter(data)
        probabilities = [count / len(data) for count in counts.values()]
        entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
        return entropy

    def extract_strings(self, data):
        strings = re.findall(b'[A-Za-z0-9]{4,}', data)
        return [s.decode('ascii', errors='ignore') for s in strings[:20]]

    def extract_imports(self, data):
        common_apis = [
            b"CreateProcess", b"WriteFile", b"RegSetValue", b"CryptEncrypt",
            b"InternetConnect", b"SetWindowsHook", b"VirtualAlloc", b"LoadLibrary"
        ]
        found_imports = []
        for api in common_apis:
            if api in data:
                found_imports.append(api.decode())
        return found_imports

    def parse_pe_header(self, data):
        if len(data) < 64:
            return {}

        try:
            dos_header = struct.unpack("<H", data[:2])[0]
            if dos_header != 0x5A4D:
                return {"valid_pe": False}

            return {
                "valid_pe": True,
                "dos_signature": hex(dos_header),
                "estimated_entry_point": hex(struct.unpack("<I", data[40:44])[0] if len(data) > 44 else 0)
            }
        except:
            return {"valid_pe": False}

    def dynamic_analysis(self, sample_data):
        api_sequence = self.simulate_api_calls(sample_data)
        network_behavior = self.simulate_network_activity()
        file_operations = self.simulate_file_operations()
        registry_changes = self.simulate_registry_operations()

        return {
            "api_calls": api_sequence,
            "network_activity": network_behavior,
            "file_operations": file_operations,
            "registry_changes": registry_changes
        }

    def simulate_api_calls(self, sample_data):
        api_patterns = {
            b"CreateProcess": ["kernel32.dll", "CreateProcessA"],
            b"CryptEncrypt": ["advapi32.dll", "CryptEncrypt"],
            b"InternetConnect": ["wininet.dll", "InternetConnectA"],
            b"RegSetValue": ["advapi32.dll", "RegSetValueExA"]
        }

        called_apis = []
        for pattern, (dll, api) in api_patterns.items():
            if pattern in sample_data:
                called_apis.append({"dll": dll, "function": api, "frequency": sample_data.count(pattern)})

        return called_apis

    def simulate_network_activity(self):
        return {
            "connections": [
                {"ip": "192.168.1.100", "port": 4444, "protocol": "TCP"},
                {"ip": "10.0.0.50", "port": 80, "protocol": "HTTP"}
            ],
            "dns_queries": ["malware-c2.example.com", "update.badsite.net"]
        }

    def simulate_file_operations(self):
        return {
            "created": ["%TEMP%\\malware.exe", "%STARTUP%\\persistence.exe"],
            "modified": ["%SYSTEM32%\\drivers\\etc\\hosts"],
            "deleted": ["shadow_copies"]
        }

    def simulate_registry_operations(self):
        return {
            "keys_created": ["HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\Malware"],
            "values_set": [{"key": "HKCU\\Software\\Policies\\Microsoft\\Windows\\System", "value": "DisableTaskMgr"}]
        }

    def build_control_flow_graph(self, api_calls):
        self.control_flow_graph.clear()

        nodes = ["START"] + [call["function"] for call in api_calls] + ["END"]
        self.control_flow_graph.add_nodes_from(nodes)

        for i in range(len(nodes) - 1):
            self.control_flow_graph.add_edge(nodes[i], nodes[i + 1])

        if len(api_calls) > 2:
            self.control_flow_graph.add_edge(nodes[2], nodes[1])

    def classify_malware_family(self, samples):
        features = []
        labels = []

        for sample_data, family in samples:
            static_info = self.static_analysis(sample_data)
            feature_vector = [
                static_info["entropy"],
                len(static_info["strings"]),
                len(static_info["imports"]),
                static_info["file_size"]
            ]
            features.append(feature_vector)
            labels.append(family)

        features = np.array(features)

        classifier = RandomForestClassifier(n_estimators=50, random_state=42)
        classifier.fit(features, labels)

        return classifier

    def similarity_analysis(self, samples):
        vectorizer = TfidfVectorizer(max_features=100)

        sample_strings = []
        for sample_data, _ in samples:
            static_info = self.static_analysis(sample_data)
            sample_strings.append(" ".join(static_info["strings"] + static_info["imports"]))

        tfidf_matrix = vectorizer.fit_transform(sample_strings)

        kmeans = KMeans(n_clusters=3, random_state=42)
        clusters = kmeans.fit_predict(tfidf_matrix.toarray())

        return clusters

    def visualize_analysis(self, samples):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        entropies = []
        families = []
        for sample_data, family in samples:
            static_info = self.static_analysis(sample_data)
            entropies.append(static_info["entropy"])
            families.append(family)

        family_counts = Counter(families)
        axes[0, 0].bar(family_counts.keys(), family_counts.values())
        axes[0, 0].set_title('Malware Family Distribution')
        axes[0, 0].set_ylabel('Count')

        axes[0, 1].hist(entropies, bins=10, alpha=0.7)
        axes[0, 1].set_title('Entropy Distribution')
        axes[0, 1].set_xlabel('Entropy')
        axes[0, 1].set_ylabel('Frequency')

        if samples:
            dynamic_info = self.dynamic_analysis(samples[0][0])
            self.build_control_flow_graph(dynamic_info["api_calls"])

            pos = nx.spring_layout(self.control_flow_graph)
            nx.draw(self.control_flow_graph, pos, ax=axes[1, 0],
                   with_labels=True, node_color='lightblue',
                   node_size=1000, font_size=8)
            axes[1, 0].set_title('Control Flow Graph')

        clusters = self.similarity_analysis(samples)
        axes[1, 1].scatter(range(len(clusters)), clusters, c=clusters, cmap='viridis')
        axes[1, 1].set_title('Similarity Clustering')
        axes[1, 1].set_xlabel('Sample Index')
        axes[1, 1].set_ylabel('Cluster')

        plt.tight_layout()
        plt.show()

analyzer = MalwareAnalyzer()

samples = []
for malware_type in ["trojan", "ransomware", "rootkit", "botnet"]:
    for _ in range(25):
        sample, family = analyzer.generate_malware_sample(malware_type)
        samples.append((sample, family))

print(f"Generated {len(samples)} malware samples")

sample_analysis = analyzer.static_analysis(samples[0][0])
print(f"Sample entropy: {sample_analysis['entropy']:.2f}")
print(f"Detected imports: {sample_analysis['imports']}")

dynamic_results = analyzer.dynamic_analysis(samples[0][0])
print(f"API calls detected: {len(dynamic_results['api_calls'])}")
print(f"Network connections: {len(dynamic_results['network_activity']['connections'])}")

classifier = analyzer.classify_malware_family(samples)
test_sample = samples[50][0]
test_analysis = analyzer.static_analysis(test_sample)
test_features = np.array([[test_analysis["entropy"],
                          len(test_analysis["strings"]),
                          len(test_analysis["imports"]),
                          test_analysis["file_size"]]])
prediction = classifier.predict(test_features)
print(f"Classified as: {prediction[0]}")

analyzer.visualize_analysis(samples)